In [ ]:
# !uv add dspy

In [ ]:
%load_ext rich

In [ ]:
import os

import dspy

from aymurai.utils.json_data import load_json

## Fetch data

In [ ]:
labels_dir = "/workspace/notebooks/experiments/entity-disambiguation/entities-reviewed"

In [ ]:
filenames = os.listdir(labels_dir)
filenames[:1]

In [ ]:
filename = filenames[0]
label_data = load_json(os.path.join(labels_dir, filename))
label_data

## Documents

In [ ]:
import mimetypes
import time
from pathlib import Path
from typing import Iterable

import requests

In [ ]:
BASE_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://localhost:8899")
ENDPOINT = f"{BASE_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "/resources/data/restricted/summarization")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

In [ ]:
doc_path = Path(os.path.join(DATA_ROOT, "aymurai - ejemplo 01.docx"))
extracted_document = call_extraction_api(requests.Session(), doc_path)
document = "\n".join(extracted_document["detail"]["document"])
print(document)

In [ ]:
references = load_json(os.path.join(labels_dir, filename))
references

## Evaluation

In [ ]:
# lm = dspy.LM("ollama_chat/gpt-oss:20b", api_base="http://localhost:11434")
lm = dspy.LM("ollama_chat/qwen3:8b", api_base="http://localhost:11434")
dspy.configure(lm=lm)

In [ ]:
print(document)

In [ ]:
# qwen3:14b
summarize = dspy.ChainOfThought("document -> summary")
response = summarize(document=document)
print(response.summary)

In [ ]:
summarize = dspy.ChainOfThought("document -> summary")
response = summarize(document=document)
print(response.summary)

In [ ]:
print(response.reasoning)

In [ ]:
from dataclasses import dataclass

from rapidfuzz import fuzz


class AssessSummary(dspy.Signature):
    """High-precision assessment of legal summaries with strict grounding requirements."""

    # Inputs
    document = dspy.InputField(
        desc="Verbatim source document. Treat it as the only ground truth; cite exact passages when justifying decisions."
    )
    summary = dspy.InputField(
        desc="Candidate legal summary whose clarity, factual grounding, and entity coverage will be judged."
    )
    references = dspy.InputField(
        desc="Ordered list of salient legal entities, citations, or parties expected to appear verbatim in the summary."
    )

    # Outputs
    language_ok: bool = dspy.OutputField(
        desc="Return True only if the input summary is completely written in the same language as the document."
    )
    language_reason = dspy.OutputField(
        desc="Identify any language mismatches or confirm consistent language use."
    )
    clarity_ok: bool = dspy.OutputField(
        desc="Return True only if the summary is concise, logically ordered, and understandable without assuming missing context."
    )
    clarity_reason = dspy.OutputField(
        desc="Briefly highlight phrasing strengths or point out specific sentences that harmed clarity."
    )
    factfulness_ok: bool = dspy.OutputField(
        desc="Return True only if every claim is explicitly supported by the document. Any unsupported or extrapolated statement must flip this to False."
    )
    factfulness_reason = dspy.OutputField(
        desc="Quote the exact evidence backing supported claims. Explicitly list each unsupported or hallucinated statement."
    )
    coverage_ok: bool = dspy.OutputField(
        desc="Return True only if the summary mentions the expected references with correct attributes and relationships."
    )
    coverage_reason = dspy.OutputField(
        desc="Explain which references were correctly included or omitted, explicitly citing the relevant summary spans."
    )


@dataclass
class SummaryScore:
    score: float
    coverage_score: float
    judgement: dspy.Prediction


class SummaryMetric(dspy.Module):
    COVERAGE_THRESHOLD = 0.9

    def __call__(self, document, references, summary, trace=None):
        coverage_score = self._coverage_score(summary, references)

        # with dspy.context(
        #     lm=dspy.LM("ollama_chat/gpt-oss:20b", api_base="http://localhost:11434")
        # ):
        # Run assessment model
        judgement = self.predict(
            document=document,
            summary=summary,
            references=references,
        )

        if not judgement.language_ok:
            return SummaryScore(
                score=0.0,
                coverage_score=coverage_score,
                judgement=judgement,
            )

        score = sum(
            [
                0.2 * (judgement.clarity_ok or 0),
                0.2 * (judgement.factfulness_ok or 0),
                0.6 * (coverage_score * (judgement.coverage_ok or 0)),
            ]
        )

        return SummaryScore(
            score=score,
            coverage_score=coverage_score,
            judgement=judgement,
        )

    def predict(self, document, summary, references):
        return dspy.Predict(AssessSummary)(
            document=document,
            summary=summary,
            references=references,
        )

    def _coverage_score(
        self, summary: str, references, threshold: float | None = None
    ) -> float:
        """
        Calculate the coverage score of the summary against the reference entities.

        Args:
            summary (str): The summary text to be evaluated for coverage.
            references (list): The list of reference entities expected to be covered in the summary.
            threshold (float | None): The minimum similarity threshold to consider an entity as covered.
                Defaults to COVERAGE_THRESHOLD.

        Returns:
            float: The coverage score representing how well the summary covers the reference entities.
        """
        if not references:
            return 1.0

        threshold = threshold or self.COVERAGE_THRESHOLD
        summary_lower = summary.lower()
        total_score, counted = 0.0, 0

        for ref in references:
            aliases = list(ref.get("aliases") or [])
            canonical = ref.get("canonical_text")
            if canonical:
                aliases = [canonical, *aliases]

            aliases = set(alias.strip().lower() for alias in aliases if alias)

            best = max(
                fuzz.partial_ratio(alias.lower(), summary_lower) / 100.0
                for alias in aliases
            )
            if best < threshold:
                best = 0.0

            total_score += best
            counted += 1

        return total_score / counted if counted else 0.0


In [ ]:
metric = SummaryMetric()

In [ ]:
# phi3:14b
score = metric(document, references, response.summary)
score

In [ ]:
score = metric(document, references, response.summary)
score

In [ ]:
summarization_results = load_json(
    DATA_ROOT / "summarization-benchmark-results-cuda.json"
)

In [ ]:
len(summarization_results)

In [ ]:
summaries = [
    doc
    for doc in summarization_results
    if doc["doc_path"] == os.path.basename(doc_path)
]

len(summaries)

In [ ]:
from tqdm import tqdm

evaluation_results = []

for summary_entry in tqdm(summaries):
    summary = summary_entry["chat_response"]
    score = metric(document, references, summary)
    evaluation_results.append(
        {
            "doc_path": summary_entry["doc_path"],
            "summary": summary,
            "score": score,
            "metadata": {
                "model": summary_entry.get("model"),
                "system_prompt_type": summary_entry.get("system_prompt_type"),
                "options": summary_entry.get("options"),
            },
        }
    )

In [ ]:
evaluation_results = sorted(
    evaluation_results, key=lambda x: x["score"].score, reverse=True
)

In [ ]:
evaluation_results

In [ ]:
# Update the score recomputation

evaluation_results_bkp = evaluation_results.copy()

In [ ]:
evaluation_results[-1]["score"].score = 0

In [ ]:
evaluation_results[-1]["score"]

In [ ]:
for eval_result in evaluation_results:
    summary_score = eval_result["score"]
    judgement = summary_score.judgement

    score = sum(
        [
            0.2 * (judgement.clarity_ok or 0),
            0.2 * (judgement.factfulness_ok or 0),
            0.6 * (summary_score.coverage_score or 0),
        ]
    )

    summary_score.score = score

In [ ]:
evaluation_resultsv = sorted(
    evaluation_results, key=lambda x: x["score"].score, reverse=True
)


In [ ]:
evaluation_results

In [ ]:
lm.history[0]

---
## Optimization

In [ ]:
labels_docs_map = {}  # Fill in with appropriate mapping of labels to documents

In [ ]:
examples = []

for filename, doc_path in labels_docs_map.items():
    references = load_json(os.path.join(labels_dir, filename))
    extracted_document = call_extraction_api(requests.Session(), Path(doc_path))
    document = "\n".join(extracted_document["detail"]["document"])

    example = dspy.Example(
        document=document,
        references=references,
    ).with_inputs("document")
    examples.append(example)

In [ ]:
examples[:2]

In [ ]:
# Split examples into simple train/dev partitions
train_examples = examples[:8]
dev_examples = examples[8:]

print(f"Train examples: {len(train_examples)}")
print(f"Dev examples: {len(dev_examples)}")

In [ ]:
class CoTSummarizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought("document -> summary")

    def forward(self, document):
        # references flows through so optimizers can use it in demos/prompts
        return self.cot(document=document)

In [ ]:
metric_module = SummaryMetric()


def summary_metric(example, prediction, trace=None):
    package = metric_module(
        document=example.document,
        references=example.references,
        summary=prediction.summary,
        trace=trace,
    )

    if trace is not None:
        return package.score >= 0.75

    return package.score

In [ ]:
evaluate = dspy.Evaluate(
    devset=examples,
    metric=summary_metric,
    num_threads=8,
    display_progress=True,
    display_table=True,
)

evaluation = evaluate(
    program=CoTSummarizer(),
)

evaluation

In [ ]:
from dspy.teleprompt import COPRO

model_to_generate_prompts = dspy.LM(
    "ollama_chat/gpt-oss:20b",
    api_base="http://localhost:11434",
    drop_params=True,
)

copro = COPRO(
    prompt_model=model_to_generate_prompts,
    metric=summary_metric,
)
eval_kwargs = {"num_threads": 16, "display_progress": True, "display_table": 0}

copro_program = copro.compile(
    student=CoTSummarizer(),
    trainset=examples,
    eval_kwargs=eval_kwargs,
)

copro_program

In [ ]:
copro_program

In [ ]:
from dspy.teleprompt import COPRO

model_to_generate_prompts = dspy.LM(
    "ollama_chat/gpt-oss:20b",
    api_base="http://localhost:11434",
    drop_params=True,
)

copro = COPRO(
    prompt_model=model_to_generate_prompts,
    metric=summary_metric,
)
eval_kwargs = {"num_threads": 16, "display_progress": True, "display_table": 0}

copro_program = copro.compile(
    student=CoTSummarizer(),
    trainset=examples,
    eval_kwargs=eval_kwargs,
)

copro_program

In [ ]:
copro_program

In [ ]:
evaluate = dspy.Evaluate(
    devset=examples,
    metric=summary_metric,
    num_threads=8,
    display_progress=True,
    display_table=True,
)


evaluation = evaluate(program=copro_program)
evaluation

In [ ]:
from dspy.teleprompt import MIPROv2

program = CoTSummarizer()

optimizer = MIPROv2(
    prompt_model=model_to_generate_prompts,
    metric=summary_metric,
    auto="medium",
)

# Optimize program
print("Optimizing program with MIPRO...")
optimized_program = optimizer.compile(
    program.deepcopy(),
    trainset=train_examples,
    valset=dev_examples,
    max_bootstrapped_demos=4,
    max_labeled_demos=0,
)

In [ ]:
bootstrapped_eval = evaluate(program=optimized_program)
bootstrapped_eval